In [ ]:
import h5py
import torch
import numpy as np
from safetensors.torch import load_file
from tqdm.notebook import tqdm
import pandas as pd

class FeatureFinder:
    def __init__(self, residual_stream_path, model_path, device="cuda:2"):
        self.device = torch.device(device)
        self.load_data(residual_stream_path, model_path)
        
    def load_data(self, residual_stream_path, model_path):
        """Load residual_stream, texts and SAE model"""
        print("Loading data...")
        with h5py.File(residual_stream_path, 'r') as f:
            self.residual_stream = f['residual_stream']
            self.residual_stream = self.residual_stream[:len(self.residual_stream)//4]
            self.texts = f['text_refs'][:]
        
        self.sae_state = load_file(model_path)
        
        # Move model params to device
        self.encoder_w = self.sae_state['_orig_mod.encode.weight'].to(self.device).to(torch.bfloat16)
        self.encoder_b = self.sae_state['_orig_mod.encode.bias'].to(self.device).to(torch.bfloat16)
        print(self.encoder_b.dtype)
        
    def get_feature_residual_stream(self, batch_size=4096):
        """Get feature residual_stream in batches, keeping in bfloat16 on GPU"""
        print("Computing feature residual_stream...")
        n_samples = len(self.residual_stream)
        n_features = self.encoder_w.shape[0]
        
        # Initialize on GPU in bfloat16
        all_features = torch.zeros(
            (n_samples, n_features), 
            dtype=torch.bfloat16,
            device=self.device
        )
        
        for i in tqdm(range(0, n_samples, batch_size)):
            batch_end = min(i + batch_size, n_samples)
            # Convert batch to bfloat16
            batch = torch.tensor(
                self.residual_stream[i:batch_end], 
                dtype=torch.bfloat16,
                device=self.device
            )
            
            # Forward pass through encoder
            with torch.no_grad():
                features = torch.relu(self.encoder_w @ batch.T + self.encoder_b.unsqueeze(1))
                all_features[i:batch_end] = features.T

        return all_features
    
    def analyze_feature_intervals(self, feature_residual_stream, n_intervals=13, samples_per_interval=10):
        """Analyze features across activation intervals"""
        print("Analyzing feature intervals...")
        n_features = feature_residual_stream.shape[1]
        feature_data = []
        
        for feat_idx in tqdm(range(n_features)):
            feat_residual_stream = feature_residual_stream[:, feat_idx]
            
            # Skip if feature never activates
            if feat_residual_stream.max() <= 0:
                continue
                
            # Create intervals
            max_act = feat_residual_stream.max()
            interval_bounds = np.linspace(0, max_act, n_intervals+1)
            
            # Analyze each interval
            for i in range(n_intervals):
                lower = interval_bounds[i]
                upper = interval_bounds[i+1]
                
                # Find examples in this interval
                mask = (feat_residual_stream >= lower) & (feat_residual_stream < upper)
                interval_indices = np.where(mask)[0]
                
                if len(interval_indices) == 0:
                    continue
                    
                # Sample examples from interval
                if len(interval_indices) > samples_per_interval:
                    interval_indices = np.random.choice(
                        interval_indices, 
                        samples_per_interval, 
                        replace=False
                    )
                
                # Record examples
                for idx in interval_indices:
                    feature_data.append({
                        'feature_id': feat_idx,
                        'interval': i,
                        'activation': feat_residual_stream[idx],
                        'text': self.texts[idx],
                    })
                    
        return pd.DataFrame(feature_data)
    
    def get_feature_statistics(self, feature_residual_stream):
        """Compute basic statistics for each feature"""
        stats = {
            'density': (feature_residual_stream > 0).mean(axis=0),
            'mean_activation': feature_residual_stream.mean(axis=0),
            'max_activation': feature_residual_stream.max(axis=0),
            'activation_frequency': (feature_residual_stream > 0).sum(axis=0)
        }
        return pd.DataFrame(stats)
    
    def find_interesting_features(self, feature_residual_stream, min_density=1e-4, max_density=0.1):
        """Find potentially interesting features based on activation patterns"""
        stats = self.get_feature_statistics(feature_residual_stream)
        
        # Filter features within density range
        interesting_features = stats[
            (stats['density'] >= min_density) & 
            (stats['density'] <= max_density)
        ].sort_values('max_activation', ascending=False)
        
        return interesting_features
    
    def analyze_features(self, batch_size=1024):
        """Run full feature analysis"""
        # Get feature residual_stream
        feature_residual_stream = self.get_feature_residual_stream(batch_size)
        
        # Find interesting features
        interesting_features = self.find_interesting_features(feature_residual_stream)
        print(f"\nFound {len(interesting_features)} potentially interesting features")
        
        # Get interval analysis for interesting features
        feature_examples = self.analyze_feature_intervals(feature_residual_stream)
        
        return {
            'residual_stream': feature_residual_stream,
            'interesting_features': interesting_features,
            'feature_examples': feature_examples
        }

def display_feature_examples(feature_examples, feature_id):
    """Display examples for a specific feature across activation intervals"""
    feature_data = feature_examples[feature_examples['feature_id'] == feature_id]
    
    print(f"Examples for Feature {feature_id}:")
    print("-" * 80)
    
    for interval in sorted(feature_data['interval'].unique()):
        interval_examples = feature_data[feature_data['interval'] == interval]
        
        print(f"\nInterval {interval} (activation range):")
        for _, row in interval_examples.iterrows():
            print(f"Activation: {row['activation']:.4f}")
            print(f"Text: {row['text']}")
            print("-" * 40)
            
# Usage example:
finder = FeatureFinder(
    residual_stream_path='/home/henry/Documents/PythonProjects/open-concept-steering/data/residual_stream_residual_stream_llama1b_bf16.h5',
    model_path='/home/henry/Documents/PythonProjects/open-concept-steering/out/sae_16k/final_model.safetensors',
    device="cuda:2"
)

# Run the analysis
results = finder.analyze_features()

# Look at the first few interesting features
print(results['interesting_features'].head())

# Look at examples for one of the interesting features
feature_id = results['interesting_features'].index[0]
display_feature_examples(results['feature_examples'], feature_id)